# ELIZA no Jupyter

Execute as células em ordem para acompanhar o projeto no **VS Code com Jupyter local**.

Este notebook verifica o ambiente, demonstra SQLite com dados fictícios e permite conversar com o ELIZA Classic offline. Use o Streamlit para conversar no navegador e guardar o histórico local. ELIZA é uma simulação conversacional.

Selecione o kernel `.venv` do projeto antes de executar. As dependências são instaladas pelo terminal conforme o README.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

project_root = next(
    (
        root
        for root in (Path.cwd(), Path.cwd().parent)
        if (root / "requirements.txt").is_file() and (root / "eliza").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Abra a pasta do projeto no VS Code antes de executar o notebook.")
REQUIREMENTS = [
    line.strip()
    for line in (project_root / "requirements.txt").read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.lstrip().startswith("#")
]
print("Ambiente local. Confirme que o kernel selecionado é .venv.")
print("Python:", sys.version.split()[0])
print("Interpretador:", sys.executable)

## Ambiente Python

A tabela mostra versões instaladas. Nenhuma chave é lida ou exibida.

In [ ]:
import sqlite3

from IPython.display import Markdown, display

rows = ["| Biblioteca | Versão |", "|---|---|"]
for requirement in REQUIREMENTS:
    name = requirement.split("==")[0]
    rows.append(f"| {name} | {version(name)} |")
rows.append(f"| SQLite (incluído no Python) | {sqlite3.sqlite_version} |")
display(Markdown("\n".join(rows)))

## Primeiro exemplo com SQLite

Este exemplo usa **SQLAlchemy + SQLite** em um arquivo temporário. Ele grava duas mensagens, fecha a conexão, reabre o banco e mostra os registros recuperados. O arquivo é apagado automaticamente ao finalizar a célula.

As mensagens são fictícias e previamente definidas; não são respostas geradas pelo chatbot.

In [ ]:
import tempfile
from pathlib import Path

from sqlalchemy import URL, create_engine, text

with tempfile.TemporaryDirectory(prefix="eliza-notebook-") as directory:
    database_url = URL.create("sqlite", database=str(Path(directory) / "demo.sqlite3"))
    engine = create_engine(database_url)
    try:
        with engine.begin() as connection:
            connection.execute(
                text("CREATE TABLE messages (id INTEGER PRIMARY KEY, role TEXT, content TEXT)")
            )
            connection.execute(
                text("INSERT INTO messages (role, content) VALUES (:role, :content)"),
                [
                    {"role": "Usuário", "content": "Estou preocupado com meu futuro."},
                    {"role": "ELIZA (exemplo)", "content": "O que mais preocupa você?"},
                ],
            )
    finally:
        engine.dispose()

    reopened = create_engine(database_url)
    try:
        with reopened.connect() as connection:
            messages = connection.execute(
                text("SELECT role, content FROM messages ORDER BY id")
            ).all()
        rows = ["| Participante | Mensagem recuperada |", "|---|---|"]
        rows.extend(f"| {role} | {content} |" for role, content in messages)
        display(Markdown("\n".join(rows)))
        assert len(messages) == 2
        print("Gravação e recuperação do arquivo SQLite: OK")
    finally:
        reopened.dispose()

## Bibliotecas opcionais

Google Gen AI será usado na integração AI; ChromaDB será usado na memória semântica. A tabela informa apenas se os pacotes estão instalados. Esta célula não chama APIs, não baixa modelos e não testa a qualidade da recuperação semântica.

In [ ]:
rows = ["| Componente | Estado neste ambiente |", "|---|---|"]
for package in ("google-genai", "chromadb"):
    try:
        status = f"Instalado: {version(package)}"
    except PackageNotFoundError:
        status = "Opcional; ainda não instalado"
    rows.append(f"| {package} | {status} |")
display(Markdown("\n".join(rows)))

## Como o Classic responde

O mecanismo reconhece padrões, troca pronomes e seleciona respostas previamente escritas. A regra acionada aparece abaixo de cada resposta para você acompanhar o funcionamento.

É uma recriação inspirada no conceito DOCTOR, sem reproduzir integralmente as regras históricas. Não há compreensão humana nem geração por IA.

**No VS Code:** basta executar a próxima célula com o kernel `.venv`.


In [ ]:
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from eliza import ClassicEliza  # noqa: E402 -- caminho do pacote preparado acima
from eliza.reflections import reflect  # noqa: E402

print(ClassicEliza().welcome_message)

### Exemplo de conversa

Cada instância corresponde a uma conversa. Ao repetir esta célula, o exemplo começa de novo. A referência ao tema anterior é uma memória textual curta, e não uma interpretação do que foi dito.

In [ ]:
exemplo_pt = ClassicEliza()
for mensagem in (
    "Olá!",
    "Estou preocupado com meu futuro.",
    "Tenho medo de fracassar.",
    "Ainda penso nisso.",
    "Talvez.",
    "Tchau!",
):
    resposta = exemplo_pt.reply(mensagem)
    print("Você:", mensagem)
    print("ELIZA:", resposta.text)
    print("Regra:", resposta.rule_id, "\n")

### Como os pronomes mudam

As substituições são feitas em uma única passagem para evitar, por exemplo, transformar `meu → seu → meu`. A transformação é heurística, sem análise gramatical completa.


In [ ]:
for texto in ("minha mãe e meu futuro", "eu estou preocupado", "minhas escolhas"):
    print(f"{texto} → {reflect(texto)}")

### Variações e respostas genéricas

As respostas de uma mesma regra giram em ordem fixa. Repetir a mesma mensagem aciona uma pergunta específica sobre repetição. Quando não encontra um padrão, ELIZA usa uma pergunta genérica ou retoma o último tema capturado.

In [ ]:
demonstracao = ClassicEliza()
for mensagem in (
    "Estou preocupado com meu futuro.",
    "Estou preocupado com meu trabalho.",
    "Estou preocupado com meu trabalho.",
    "Hmm.",
    "Talvez.",
    "Não quero fracassar.",
):
    resposta = demonstracao.reply(mensagem)
    print(f"Você: {mensagem}\nELIZA [{resposta.rule_id}]: {resposta.text}\n")

## Converse com a ELIZA

Execute a célula de criação uma vez. Depois altere **somente `minha_mensagem`** na célula seguinte e execute-a com `Shift+Enter` a cada interação. O histórico e a rotação permanecem na memória do kernel.

Para começar outra conversa, execute novamente a célula de criação ou use `minha_eliza.reset()`. Reiniciar o kernel apaga a sessão. Esta conversa livre fica somente na memória. A seção SQLite a seguir demonstra como salvar uma conversa no banco.

In [ ]:
minha_eliza = ClassicEliza()
print(minha_eliza.welcome_message)

In [ ]:
minha_mensagem = "Estou preocupado com meu futuro."

try:
    resposta = minha_eliza.reply(minha_mensagem)
except (TypeError, ValueError) as error:
    print("Confira a mensagem:", error)
else:
    for turno in minha_eliza.history:
        print("Você:", turno.user)
        print("ELIZA:", turno.assistant)
        print()
    print("Última regra:", resposta.rule_id)

## Histórico de conversas

Agora usaremos a camada real de persistência do projeto: `ConversationRepository` guarda conversas e mensagens; `ConversationMemory` conecta o histórico ao Classic e reconstrói o contexto ao retomar.

A demonstração usa um **arquivo SQLite temporário, com mensagens fictícias**. Ele permanece entre as células até a limpeza final. Nada é gravado no seu banco permanente em `data/`, e nenhuma conversa pessoal é excluída.

Execute as próximas células em ordem no kernel `.venv`.


In [ ]:
from database import ConversationRepository
from eliza.memory import ConversationMemory

print("Camadas de persistência e memória importadas.")

### Criar uma conversa

O identificador permite recuperar esta conversa depois. O modo, o idioma e as datas UTC ficam armazenados junto do título. Reexecutar esta célula fecha e remove apenas a demonstração temporária anterior.

In [ ]:
import tempfile

repositorio_anterior = globals().get("repositorio_demo")
if repositorio_anterior is not None:
    repositorio_anterior.close()
pasta_anterior = globals().get("pasta_banco_demo")
if pasta_anterior is not None:
    pasta_anterior.cleanup()

pasta_banco_demo = tempfile.TemporaryDirectory(prefix="eliza-historico-")
caminho_demo = Path(pasta_banco_demo.name) / "conversas.sqlite3"
repositorio_demo = ConversationRepository(caminho_demo)
memoria_demo = ConversationMemory(repositorio_demo)
conversa_demo = repositorio_demo.create_conversation("Preocupações com o futuro")

print("Título:", conversa_demo.title)
print("Identificador:", conversa_demo.id)
print("Modo:", conversa_demo.mode)
print("Idioma:", conversa_demo.language)
print("Criada em UTC:", conversa_demo.created_at.isoformat())
print("Arquivo SQLite criado:", caminho_demo.is_file())
assert repositorio_demo.get_history(conversa_demo.id).messages == ()

### Salvar mensagens

Cada chamada `send` gera a resposta Classic e salva **as duas mensagens na mesma transação**. Se a gravação falhar, nenhuma metade do turno fica no histórico.

In [ ]:
for mensagem in ("Estou preocupado com meu futuro.", "Tenho medo de fracassar."):
    turno_salvo = memoria_demo.send(conversa_demo.id, mensagem)
    print("Você:", turno_salvo.user.content)
    print("ELIZA:", turno_salvo.assistant.content)
    print("Regra salva:", turno_salvo.assistant.rule_id, "\n")

historico_antes_de_fechar = repositorio_demo.get_history(conversa_demo.id)
print("Mensagens gravadas:", len(historico_antes_de_fechar.messages))
print("Turnos completos:", historico_antes_de_fechar.conversation.revision)

### Retomar a conversa

O objeto anterior é fechado e um novo repositório abre o mesmo arquivo. O histórico volta do disco. A próxima mensagem pode retomar o tema, porque o estado Classic é reconstruído a partir das entradas anteriores.

In [ ]:
repositorio_demo.close()
repositorio_demo = ConversationRepository(caminho_demo)
memoria_demo = ConversationMemory(repositorio_demo)
recuperada = repositorio_demo.get_history(conversa_demo.id)
assert recuperada == historico_antes_de_fechar

for mensagem_salva in recuperada.messages:
    participante = "Você" if mensagem_salva.role == "user" else "ELIZA"
    print(f"{mensagem_salva.position}: {participante}: {mensagem_salva.content}")
print("\nHistórico recuperado após reabrir: OK")
continuacao = memoria_demo.send(conversa_demo.id, "Ainda penso nisso.")
print("\nVocê:", continuacao.user.content)
print("ELIZA:", continuacao.assistant.content)
print("Regra:", continuacao.assistant.rule_id)

### Listar conversas salvas

A lista vem da conversa atualizada mais recentemente para a mais antiga. A segunda conversa tem contexto e mensagens próprios.

In [ ]:
conversa_preservada = repositorio_demo.create_conversation("Outra conversa")
memoria_demo.send(conversa_preservada.id, "Olá!")
for item in repositorio_demo.list_conversations():
    print(f"{item.title} | {item.mode} | {item.revision} turno(s) | {item.id}")
assert len(repositorio_demo.list_conversations()) == 2

### Excluir uma conversa

Esta célula apaga **a primeira conversa fictícia do banco temporário**, com suas mensagens. A segunda fica intacta. Ao final, fecha o repositório e remove a pasta temporária da demonstração.

Para repetir a experiência, volte à célula **Criar uma conversa** e execute em ordem novamente.

In [ ]:
assert repositorio_demo.delete_conversation(conversa_demo.id)
assert repositorio_demo.get_conversation(conversa_demo.id) is None
assert len(repositorio_demo.get_history(conversa_preservada.id).messages) == 2
print("Conversa escolhida e suas mensagens excluídas.")
print("Conversa preservada:", repositorio_demo.list_conversations()[0].title)
print("Excluir novamente retorna:", repositorio_demo.delete_conversation(conversa_demo.id))

repositorio_demo.close()
pasta_banco_demo.cleanup()
print("Demonstração concluída. Arquivos temporários removidos.")

### Guardar suas conversas

Para guardar suas próprias conversas, a API usa `ConversationRepository.from_settings()` e salva por padrão em `data/eliza.db`, fora do Git. O exemplo abaixo é uma receita; não é executado automaticamente por este notebook.

```python
with ConversationRepository.from_settings() as repositorio:
    conversa = repositorio.create_conversation("Minha conversa")
    memoria = ConversationMemory(repositorio)
    resposta = memoria.send(conversa.id, "Estou preocupado com meu futuro.")
    print(resposta.assistant.content)
    print("Guarde este identificador:", conversa.id)
```

Em uma sessão posterior, abra novamente o repositório, use `list_conversations()` para escolher um identificador e `get_history(id)` para ler ou `ConversationMemory(repositorio).send(id, mensagem)` para continuar. Use `delete_conversation(id)` somente para a conversa que deseja remover.
